# 05 — Exploratory pooled-cell analysis (Good vs Poor)

**Scope: exploratory and descriptive. This notebook does not produce a confirmatory
outcome test.**

CellPhoneDB was run once on all pooled Good cells and once on all pooled Poor cells.
That construction collapses 13 Good patients into one "super-patient" and 11 Poor
patients into another. Every p-value produced below therefore counts *cell-type pairs*
or *ligand–receptor interactions* as its unit of replication — **not patients**. It
cannot support a statement about a patient-level clinical association, and it is not
an independent replication of, or a validation of, the patient-level analysis in
`04_patient_level_inference.ipynb`.

Every inferential table written here carries `analysis_scope = 'exploratory_pooled'`.

What this notebook does, in order:

1. **Prespecification** — write the primary pooled table name and thresholds to the run
   log **before** any results are computed.
2. **Differential enrichment** — port of `legacy_sources/cpdb_differential_enrichment.py`
   (paired Wilcoxon signed-rank; logic untouched).
3. **Per-sample consistency** — port of `legacy_sources/cpdb_persample_consistency.py`
   with the hardcoded `model_outcome`-derived sample lists removed and replaced by a
   runtime derivation from `obs['clinical_outcome']`.
4. **Concordance annotation** — join to the finalised patient-level table, for
   annotation only.

Reporting language is constrained: results are "associated with clinical-outcome group"
/ "exploratory pooled-cell signal". The words *caused*, *protective*, *drives outcome*,
*validated*, and *trend* are not used.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


## Prespecification (read before running)

Four Wilcoxon tables are produced by the differential-enrichment step:

| file | unit tested | paired across |
|---|---|---|
| `wilcoxon_per_interaction_scores.csv` | ligand–receptor interaction | cell-type pairs |
| `wilcoxon_per_interaction_means.csv`  | ligand–receptor interaction | cell-type pairs |
| `wilcoxon_per_cellpair_scores.csv`    | cell-type pair             | interactions |
| `wilcoxon_per_cellpair_means.csv`     | cell-type pair             | interactions |

Choosing, per hit, whichever of these four happens to be significant is a
forking-paths problem. **`wilcoxon_per_interaction_scores.csv` is prespecified as the
primary pooled table** — it is the table the consistency step consumes and the only one
used for concordance annotation. The other three are reported as secondary descriptive
tables. This choice is written to the run log in the cell below, before any numbers are
computed.

### Multiplicity accounting — stated explicitly

Each of the four tables is Benjamini–Hochberg corrected **independently, once, within
its own file**. There is no correction across the four families. In the previous
(`model_outcome`, 17 vs 7) run the four family sizes were:

| table | family size (previous run) |
|---|---|
| `wilcoxon_per_interaction_scores.csv` | 280 |
| `wilcoxon_per_interaction_means.csv`  | 1145 |
| `wilcoxon_per_cellpair_scores.csv`    | 321 |
| `wilcoxon_per_cellpair_means.csv`     | 324 |

`detailed_interaction_summary.csv` (9,571 rows in the previous run) receives **no
multiplicity correction at all** — it is a flat descriptive listing of every
interaction × cell-type pair that CellPhoneDB called significant (p < 0.05) in at least
one group. It carries CellPhoneDB's own per-cell-pair permutation p-values and must not
be read as a corrected result table.

The family sizes for *this* run are recorded to the run log and asserted to be non-empty;
they are not asserted to equal the previous run's values, because the groups have changed
from 17/7 (`model_outcome`) to 13/11 (`clinical_outcome`).

In [ ]:
# =============================================================================
# CONFIGURATION — the only cell that defines paths, thresholds or prespecified
# choices. Every analytical value is read from config.yaml, which is the single
# source of truth; nothing below this cell introduces a new path or threshold.
# =============================================================================
from pathlib import Path
import sys

NOTEBOOK = "05_pooled_exploratory_analysis"

CODE_DIR    = Path(f"{ACC_DATA_ROOT}/scripts/named_scripts/"
                   "cellphoneDB_scripts/clinical_outcome_v2")
CONFIG_PATH = CODE_DIR / "config.yaml"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from utils import load_config                                        # noqa: E402

CFG = load_config(CONFIG_PATH)

PROJECT_ROOT = Path(CFG["project_root"])
OUTPUT_ROOT  = Path(CFG["output_root"])
assert Path(CFG["code_root"]) == CODE_DIR, \
    f"config code_root {CFG['code_root']} != {CODE_DIR}"

# --- CellPhoneDB run directories (written by 02_run_cellphonedb.ipynb) -------
CPDB_RUNS        = OUTPUT_ROOT / "cpdb_runs"
GOOD_DIR         = CPDB_RUNS / "pooled" / "good"     # balanced pooled Good, seed 0
POOR_DIR         = CPDB_RUNS / "pooled" / "poor"     # balanced pooled Poor, seed 0
PER_PATIENT_DIR  = CPDB_RUNS / "per_patient"         # one sub-directory per patient

# Downsampling-seed runs. Seed 0 is the main exploratory run (GOOD_DIR/POOR_DIR);
# seeds 1-4 are the stability runs. 06_sensitivity_analysis.ipynb discovers the
# per-seed tables at exploratory_pooled/seed_{s}/wilcoxon_per_interaction_scores.csv,
# so that layout is a contract, not a preference.
POOLED_SEEDS = [int(s) for s in CFG["pooled_downsample_seeds"]]
assert POOLED_SEEDS[0] == 0, "seed 0 must be the main pooled run"
POOLED_SEED_RUN_DIRS = {0: {"good": GOOD_DIR, "poor": POOR_DIR}}
POOLED_SEED_RUN_DIRS.update({
    s: {"good": CPDB_RUNS / f"pooled_seed{s}" / "good",
        "poor": CPDB_RUNS / f"pooled_seed{s}" / "poor"}
    for s in POOLED_SEEDS[1:]
})
SEED_DE_DIRNAME = "seed_{s}"          # consumed by 06_
SEED_DE_TABLE   = "wilcoxon_per_interaction_scores.csv"

# --- Output directories -----------------------------------------------------
EXPLORATORY_DIR         = OUTPUT_ROOT / "exploratory_pooled"
DE_OUTPUT_DIR           = EXPLORATORY_DIR / "differential_enrichment_results"
CONSISTENCY_OUTPUT_DIR  = EXPLORATORY_DIR / "consistency_results"
LOG_DIR                 = OUTPUT_ROOT / "logs"
RUN_LOG                 = LOG_DIR / f"{NOTEBOOK}_run_log.jsonl"
SUCCESS_JSON            = EXPLORATORY_DIR / "05_SUCCESS.json"
MANIFEST_PATH           = EXPLORATORY_DIR / "manifest.json"   # final MANIFEST.sha256 is written by 08_

# --- Upstream tables consumed (read-only) -----------------------------------
PRIMARY_RESULTS_CSV = OUTPUT_ROOT / "primary" / "patient_level_primary_results.csv"

# --- Thresholds — read from config.yaml, never re-declared ------------------
FDR_THRESHOLD    = float(CFG["fdr_alpha"])            # BH threshold per Wilcoxon family
CPDB_PVAL_CUTOFF = float(CFG["cpdb_nominal_pvalue"])  # CPDB p-value cutoff for "present"
SAMPLE_COL       = CFG["sample_column"]
OUTCOME_COL      = CFG["outcome_column"]
OUTCOME_LABELS   = list(CFG["outcomes"])

# The ported legacy Wilcoxon functions call multipletests(method='fdr_bh')
# literally. They are validated and must not be edited, so instead assert that
# the configured method still agrees with what that code does.
assert CFG["fdr_method"] == "fdr_bh", (
    f"config fdr_method is {CFG['fdr_method']!r}; the ported legacy differential-"
    "enrichment functions hardcode 'fdr_bh' and must not be edited. Reconcile "
    "before running.")
assert OUTCOME_COL == "clinical_outcome", \
    f"outcome column must be clinical_outcome, got {OUTCOME_COL!r}"
assert sorted(OUTCOME_LABELS) == ["Good", "Poor"]

# --- Prespecified primary pooled table --------------------------------------
PRIMARY_POOLED_TABLE = "wilcoxon_per_interaction_scores.csv"
ANALYSIS_SCOPE       = "exploratory_pooled"

# --- Expected group structure -----------------------------------------------
# These are dataset facts established and asserted in 00_audit_inputs_and_sources,
# NOT configuration keys: config.yaml deliberately contains no cohort counts.
# They are re-asserted here against the runtime derivation from clinical_outcome.
EXPECTED_N_GOOD     = 13
EXPECTED_N_POOR     = 11
EXPECTED_N_PATIENTS = 24
assert EXPECTED_N_GOOD + EXPECTED_N_POOR == EXPECTED_N_PATIENTS

# --- Family sizes observed in the previous (model_outcome, 17/7) run --------
# Recorded for documentation only. Not asserted; the groups have changed.
LEGACY_FAMILY_SIZES = {
    "wilcoxon_per_interaction_scores.csv": 280,
    "wilcoxon_per_interaction_means.csv":  1145,
    "wilcoxon_per_cellpair_scores.csv":    321,
    "wilcoxon_per_cellpair_means.csv":     324,
    "detailed_interaction_summary.csv":    9571,   # UNCORRECTED
}
BH_CORRECTED_TABLES = [
    "wilcoxon_per_interaction_scores.csv",
    "wilcoxon_per_interaction_means.csv",
    "wilcoxon_per_cellpair_scores.csv",
    "wilcoxon_per_cellpair_means.csv",
]
UNCORRECTED_TABLES = ["detailed_interaction_summary.csv"]

# --- Ordering guard ---------------------------------------------------------
# Consistency MUST be computed against a differential-enrichment output that is
# already final. Enforced by comparing file mtimes; see the ordering-hazard cell.
ENFORCE_DE_BEFORE_CONSISTENCY = True

for _d in (DE_OUTPUT_DIR, CONSISTENCY_OUTPUT_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# IMPORTS + shared-utility bindings
# =============================================================================
import json
import os
import sys
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# CFG was loaded in the configuration cell; only the remaining helpers are
# imported here. Nothing in this cell re-reads or re-declares configuration.
from utils import (sha256_file, write_manifest, log_jsonl,           # noqa: E402
                   derive_patient_outcomes, safe_unlink)


def _call_flex(fn, *variants):
    '''Call `fn` with the first argument shape it accepts.

    A thin adapter over the shared helpers so this notebook is tolerant of the
    exact argument order in utils.py, without reimplementing any of its logic.
    '''
    last = None
    for args, kwargs in variants:
        try:
            return fn(*args, **kwargs)
        except TypeError as exc:
            last = exc
    raise last


CFG_SHA = sha256_file(CONFIG_PATH)


def _log(**record):
    '''Append one JSON record to the notebook run log.'''
    rec = {"notebook": NOTEBOOK,
           "timestamp_utc": datetime.now(timezone.utc).isoformat(),
           "config_sha256": CFG_SHA,
           **record}
    _call_flex(log_jsonl,
               ((RUN_LOG, rec), {}),
               ((rec, RUN_LOG), {}),
               ((), {"path": RUN_LOG, "record": rec}))
    return rec


RUN_START = time.time()
RUN_START_ISO = datetime.now(timezone.utc).isoformat()

assert GOOD_DIR.is_dir(), f"missing pooled Good run: {GOOD_DIR}"
assert POOR_DIR.is_dir(), f"missing pooled Poor run: {POOR_DIR}"
assert PER_PATIENT_DIR.is_dir(), f"missing per-patient runs: {PER_PATIENT_DIR}"

In [ ]:
# =============================================================================
# PRESPECIFICATION RECORD — written BEFORE any result is computed.
# =============================================================================
PRESPEC = {
    "event": "prespecification",
    "run_start_utc": RUN_START_ISO,
    "analysis_scope": ANALYSIS_SCOPE,
    "confirmatory": False,
    "unit_of_replication": "cell_type_pair_or_interaction (NOT patient)",
    "primary_pooled_table": PRIMARY_POOLED_TABLE,
    "secondary_pooled_tables": [t for t in BH_CORRECTED_TABLES
                                if t != PRIMARY_POOLED_TABLE],
    "uncorrected_tables": UNCORRECTED_TABLES,
    "bh_correction": "Benjamini-Hochberg, applied once independently within each "
                     "of the four Wilcoxon files; no correction across files; "
                     "detailed_interaction_summary.csv is uncorrected",
    "fdr_threshold": FDR_THRESHOLD,
    "cpdb_pval_cutoff": CPDB_PVAL_CUTOFF,
    "config_derived_values": {k: CFG[k] for k in
                              ("fdr_alpha", "fdr_method", "cpdb_nominal_pvalue",
                               "outcome_column", "sample_column", "outcomes",
                               "pooled_downsample_seeds")},
    "good_dir": str(GOOD_DIR),
    "poor_dir": str(POOR_DIR),
    "per_patient_dir": str(PER_PATIENT_DIR),
    "legacy_family_sizes_previous_run": LEGACY_FAMILY_SIZES,
    "patient_lists": "derived at runtime from obs['clinical_outcome']; never hardcoded",
    "expected_split": {"Good": EXPECTED_N_GOOD, "Poor": EXPECTED_N_POOR},
    "pooled_hits_feed_primary_family": False,
    "claim_of_independent_replication": False,
}
_log(**PRESPEC)
PRESPEC

---
## Part 1 — Differential enrichment (paired Wilcoxon signed-rank)

Port of `legacy_sources/cpdb_differential_enrichment.py`. That script is already
argparse-driven with no hardcoded paths and no hardcoded sample lists, so this is a
config-cell swap only.

**The paired-Wilcoxon logic below is reproduced verbatim and must not be modified.**
The pairing (same cell-type pair observed in both groups), the "drop pairs where both
groups are zero" rule, the minimum of 10 non-zero differences, the log2 fold-change
pseudocount of 1e-6, and the per-file BH correction are all as validated.

The only additions are (a) an `analysis_scope` column on every written table and
(b) run-log records.

In [ ]:
# =============================================================================
# PORTED VERBATIM from legacy_sources/cpdb_differential_enrichment.py
# Statistical logic unchanged. Do not edit.
# =============================================================================

def load_cpdb_matrix(filepath):
    '''Load a CellPhoneDB output CSV (pvalues, means, interaction_scores).'''
    return pd.read_csv(filepath)


def get_metadata_and_cellpair_cols(df):
    '''Cell-type pair columns contain "|". Everything else is metadata.'''
    meta_cols = [c for c in df.columns if '|' not in c]
    cp_cols   = [c for c in df.columns if '|' in c]
    return meta_cols, cp_cols


def align_matrices(good_df, poor_df):
    '''Align two CellPhoneDB matrices to shared interactions (rows) and shared
    cell-type pairs (columns).'''
    _, cp_good = get_metadata_and_cellpair_cols(good_df)
    _, cp_poor = get_metadata_and_cellpair_cols(poor_df)

    shared_ids = set(good_df['id_cp_interaction']) & set(poor_df['id_cp_interaction'])
    shared_cps = sorted(set(cp_good) & set(cp_poor))

    print(f"    Interactions  - good: {len(good_df)}, poor: {len(poor_df)}, shared: {len(shared_ids)}")
    print(f"    Cell-type pairs - good: {len(cp_good)}, poor: {len(cp_poor)}, shared: {len(shared_cps)}")

    good_aligned = (good_df[good_df['id_cp_interaction'].isin(shared_ids)]
                    .set_index('id_cp_interaction').sort_index())
    poor_aligned = (poor_df[poor_df['id_cp_interaction'].isin(shared_ids)]
                    .set_index('id_cp_interaction').sort_index())

    return good_aligned, poor_aligned, shared_cps


def get_meta(df, interaction_id):
    '''Pull scalar metadata fields for one interaction row.'''
    meta_cols = [c for c in df.columns if '|' not in c]
    return {col: df.loc[interaction_id, col] for col in meta_cols}


def log2fc(mean_good, mean_poor, pseudocount=1e-6):
    '''log2((mean_good + eps) / (mean_poor + eps)). Positive = higher in good.'''
    return np.log2((mean_good + pseudocount) / (mean_poor + pseudocount))


def wilcoxon_per_interaction(good_mat, poor_mat, cellpair_cols, label='score'):
    '''Paired Wilcoxon signed-rank per interaction, across cell-type pairs.'''
    results = []

    for iid in good_mat.index:
        g = good_mat.loc[iid, cellpair_cols].values.astype(float)
        p = poor_mat.loc[iid, cellpair_cols].values.astype(float)

        # Drop pairs where both groups are zero
        keep = ~((g == 0) & (p == 0))
        g, p = g[keep], p[keep]

        diffs = g - p
        n_nonzero = np.sum(diffs != 0)
        if n_nonzero < 10:
            continue

        try:
            stat, pval = wilcoxon(g, p, alternative='two-sided')
        except ValueError:
            continue

        mean_g = np.mean(g)
        mean_p = np.mean(p)
        median_diff = np.median(diffs)
        lfc = log2fc(mean_g, mean_p)

        meta = get_meta(good_mat, iid)
        results.append({
            'id_cp_interaction':        iid,
            'interacting_pair':         meta.get('interacting_pair', ''),
            'classification':           meta.get('classification', ''),
            'directionality':           meta.get('directionality', ''),
            'secreted':                 meta.get('secreted', ''),
            f'mean_{label}_good':       mean_g,
            f'mean_{label}_poor':       mean_p,
            'log2fc_good_vs_poor':      lfc,
            'median_diff_good_minus_poor': median_diff,
            'n_cellpair_tested':        int(np.sum(keep)),
            'n_nonzero_diffs':          int(n_nonzero),
            'wilcoxon_stat':            stat,
            'pvalue':                   pval,
            'direction':                'higher_in_good' if lfc > 0 else 'higher_in_poor',
        })

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)
    _, fdr, _, _ = multipletests(df['pvalue'].values, method='fdr_bh')
    df['fdr'] = fdr
    return df.sort_values('pvalue').reset_index(drop=True)


def wilcoxon_per_cellpair(good_mat, poor_mat, cellpair_cols, label='score'):
    '''Paired Wilcoxon signed-rank per cell-type pair, across interactions.'''
    results = []

    for cp in cellpair_cols:
        g = good_mat[cp].values.astype(float)
        p = poor_mat[cp].values.astype(float)

        keep = ~((g == 0) & (p == 0))
        g, p = g[keep], p[keep]

        diffs = g - p
        n_nonzero = np.sum(diffs != 0)
        if n_nonzero < 10:
            continue

        try:
            stat, pval = wilcoxon(g, p, alternative='two-sided')
        except ValueError:
            continue

        mean_g = np.mean(g)
        mean_p = np.mean(p)
        median_diff = np.median(diffs)
        lfc = log2fc(mean_g, mean_p)

        results.append({
            'cell_type_pair':              cp,
            f'mean_{label}_good':          mean_g,
            f'mean_{label}_poor':          mean_p,
            'log2fc_good_vs_poor':         lfc,
            'median_diff_good_minus_poor': median_diff,
            'n_interactions_tested':       int(np.sum(keep)),
            'n_nonzero_diffs':             int(n_nonzero),
            'wilcoxon_stat':               stat,
            'pvalue':                      pval,
            'direction':                   'higher_in_good' if lfc > 0 else 'higher_in_poor',
        })

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)
    _, fdr, _, _ = multipletests(df['pvalue'].values, method='fdr_bh')
    df['fdr'] = fdr
    return df.sort_values('pvalue').reset_index(drop=True)


def create_detailed_summary(good_pvals, poor_pvals, good_scores, poor_scores,
                            good_means, poor_means, cellpair_cols,
                            cpdb_pval_threshold=0.05):
    '''Flat table of every interaction x cell-type pair CellPhoneDB called
    significant (p < threshold) in at least one group. NOT multiplicity corrected.'''
    records = []
    meta_cols = [c for c in good_pvals.columns if '|' not in c]

    for iid in good_pvals.index:
        iname = good_pvals.loc[iid, 'interacting_pair'] if 'interacting_pair' in meta_cols else iid
        cls   = good_pvals.loc[iid, 'classification']   if 'classification'   in meta_cols else ''

        for cp in cellpair_cols:
            pv_g = float(good_pvals.loc[iid, cp])
            pv_p = float(poor_pvals.loc[iid, cp])
            sig_g = pv_g < cpdb_pval_threshold
            sig_p = pv_p < cpdb_pval_threshold

            if not sig_g and not sig_p:
                continue

            sc_g = float(good_scores.loc[iid, cp])
            sc_p = float(poor_scores.loc[iid, cp])
            mn_g = float(good_means.loc[iid, cp])
            mn_p = float(poor_means.loc[iid, cp])

            if sig_g and sig_p:
                status = 'both'
            elif sig_g:
                status = 'good_only'
            else:
                status = 'poor_only'

            records.append({
                'id_cp_interaction':          iid,
                'interacting_pair':           iname,
                'classification':             cls,
                'cell_type_pair':             cp,
                'cpdb_pval_good':             pv_g,
                'cpdb_pval_poor':             pv_p,
                'sig_in_good':                sig_g,
                'sig_in_poor':                sig_p,
                'significance_status':        status,
                'interaction_score_good':     sc_g,
                'interaction_score_poor':     sc_p,
                'score_log2fc_good_vs_poor':  log2fc(sc_g, sc_p),
                'mean_expr_good':             mn_g,
                'mean_expr_poor':             mn_p,
                'mean_log2fc_good_vs_poor':   log2fc(mn_g, mn_p),
            })

    return pd.DataFrame(records)

In [ ]:
# =============================================================================
# Load and align the pooled seed-0 Good / Poor CellPhoneDB runs
# =============================================================================
_required = ['pvalues.csv', 'means.csv', 'interaction_scores.csv']
for _d in (GOOD_DIR, POOR_DIR):
    for _f in _required:
        assert (_d / _f).is_file(), f"missing required CellPhoneDB output: {_d / _f}"

good_pvals  = load_cpdb_matrix(GOOD_DIR / 'pvalues.csv')
poor_pvals  = load_cpdb_matrix(POOR_DIR / 'pvalues.csv')
good_means  = load_cpdb_matrix(GOOD_DIR / 'means.csv')
poor_means  = load_cpdb_matrix(POOR_DIR / 'means.csv')
good_scores = load_cpdb_matrix(GOOD_DIR / 'interaction_scores.csv')
poor_scores = load_cpdb_matrix(POOR_DIR / 'interaction_scores.csv')

print("  pvalues:")
good_pvals,  poor_pvals,  cp_pvals  = align_matrices(good_pvals,  poor_pvals)
print("  means:")
good_means,  poor_means,  cp_means  = align_matrices(good_means,  poor_means)
print("  interaction_scores:")
good_scores, poor_scores, cp_scores = align_matrices(good_scores, poor_scores)

shared_cps = sorted(set(cp_pvals) & set(cp_means) & set(cp_scores))

assert len(shared_cps) > 0, "no cell-type pair columns shared across all three matrices"
assert len(good_scores) > 0, "no shared interactions between pooled Good and Poor"
assert good_scores.index.equals(poor_scores.index), "score matrices are not row-aligned"
assert good_means.index.equals(poor_means.index), "mean matrices are not row-aligned"
assert good_pvals.index.equals(poor_pvals.index), "pvalue matrices are not row-aligned"

_log(event="pooled_alignment",
     n_shared_interactions=int(len(good_scores)),
     n_shared_cellpairs=int(len(shared_cps)),
     good_dir_sha_pvalues=sha256_file(GOOD_DIR / 'pvalues.csv'),
     poor_dir_sha_pvalues=sha256_file(POOR_DIR / 'pvalues.csv'))

print(f"\n  shared interactions: {len(good_scores)}   shared cell-type pairs: {len(shared_cps)}")

In [ ]:
# =============================================================================
# Run the four paired-Wilcoxon tests + the uncorrected detailed summary
# =============================================================================
res_1a = wilcoxon_per_interaction(good_scores, poor_scores, shared_cps, label='score')
res_1b = wilcoxon_per_interaction(good_means,  poor_means,  shared_cps, label='mean_expr')
res_2a = wilcoxon_per_cellpair(good_scores, poor_scores, shared_cps, label='score')
res_2b = wilcoxon_per_cellpair(good_means,  poor_means,  shared_cps, label='mean_expr')

detailed = create_detailed_summary(
    good_pvals, poor_pvals, good_scores, poor_scores, good_means, poor_means,
    shared_cps, CPDB_PVAL_CUTOFF,
)

DE_TABLES = {
    'wilcoxon_per_interaction_scores.csv': res_1a,
    'wilcoxon_per_interaction_means.csv':  res_1b,
    'wilcoxon_per_cellpair_scores.csv':    res_2a,
    'wilcoxon_per_cellpair_means.csv':     res_2b,
    'detailed_interaction_summary.csv':    detailed,
}

for _name, _df in DE_TABLES.items():
    assert len(_df) > 0, f"{_name} is empty; refusing to write a degenerate family"

# Every written table is stamped as exploratory pooled-cell output.
de_written = {}
for _name, _df in DE_TABLES.items():
    out = _df.copy()
    out['analysis_scope'] = ANALYSIS_SCOPE
    out['bh_corrected'] = _name in BH_CORRECTED_TABLES
    path = DE_OUTPUT_DIR / _name
    out.to_csv(path, index=False)
    de_written[_name] = path
    DE_TABLES[_name] = out

de_family_sizes = {n: int(len(d)) for n, d in DE_TABLES.items()}
de_family_sizes

In [ ]:
# =============================================================================
# Multiplicity bookkeeping — one BH correction per file, none across files.
# =============================================================================
family_report = []
for name, df in DE_TABLES.items():
    corrected = name in BH_CORRECTED_TABLES
    row = {
        'table': name,
        'family_size_this_run': int(len(df)),
        'family_size_previous_run_model_outcome': LEGACY_FAMILY_SIZES.get(name),
        'bh_corrected': corrected,
        'n_raw_p_lt_threshold': int((df['pvalue'] < FDR_THRESHOLD).sum()) if 'pvalue' in df else None,
        'n_bh_q_lt_threshold': int((df['fdr'] < FDR_THRESHOLD).sum()) if corrected else None,
        'analysis_scope': ANALYSIS_SCOPE,
    }
    family_report.append(row)

family_report_df = pd.DataFrame(family_report)
family_report_df.to_csv(DE_OUTPUT_DIR / 'multiplicity_family_report.csv', index=False)

# Assertions: BH applied exactly once per corrected family; uncorrected table
# carries no 'fdr' column of its own.
for name in BH_CORRECTED_TABLES:
    df = DE_TABLES[name]
    assert 'fdr' in df.columns, f"{name}: BH column missing"
    assert df['fdr'].notna().all(), f"{name}: BH produced NaN q-values"
    assert (df['fdr'] >= df['pvalue'] - 1e-12).all(), f"{name}: q < p, BH mis-applied"
for name in UNCORRECTED_TABLES:
    assert 'fdr' not in DE_TABLES[name].columns, \
        f"{name} must remain uncorrected; found an 'fdr' column"

_log(event="differential_enrichment_complete",
     analysis_scope=ANALYSIS_SCOPE,
     family_sizes=de_family_sizes,
     bh_corrected_tables=BH_CORRECTED_TABLES,
     uncorrected_tables=UNCORRECTED_TABLES,
     primary_pooled_table=PRIMARY_POOLED_TABLE,
     outputs={k: str(v) for k, v in de_written.items()},
     output_sha256={k: sha256_file(v) for k, v in de_written.items()})

family_report_df

---
## Per-seed pooled differential enrichment (downsampling stability inputs)

The pooled comparison downsamples each retained cell type to `min(n_good, n_poor)`
cells. Seed 0 is the main exploratory run; seeds 1–4 repeat the draw. The same paired
Wilcoxon test is run for every seed so that
`06_sensitivity_analysis.ipynb` can measure how much of the pooled picture is a
property of one particular draw of cells rather than of the outcome groups.

Only `wilcoxon_per_interaction_scores.csv` is written per seed, at
`exploratory_pooled/seed_{s}/`. That path is the discovery contract `06_` expects, and
writing a single table per seed directory keeps its glob unambiguous.

Each seed's table is BH-corrected once within its own file, exactly as the main run is.
Seed variability is a measure of sensitivity to cell downsampling. It is **not** a
measure of patient-level validity, and none of these tables is a confirmatory result.

In [ ]:
# =============================================================================
# Per-seed pooled differential enrichment (seeds 0-4)
# Seed 0 reuses the main run's inputs, so its table is identical to the primary
# pooled table; it is written into seed_0/ as well so 06_ sees a uniform layout.
# =============================================================================
assert sorted(POOLED_SEEDS) == sorted(POOLED_SEED_RUN_DIRS), \
    "config pooled_downsample_seeds and the seed run directories disagree"

seed_written, seed_report = {}, []
for seed in sorted(POOLED_SEED_RUN_DIRS):
    dirs = POOLED_SEED_RUN_DIRS[seed]
    out_dir = EXPLORATORY_DIR / SEED_DE_DIRNAME.format(s=seed)
    out_dir.mkdir(parents=True, exist_ok=True)

    if seed == 0:
        res = DE_TABLES[PRIMARY_POOLED_TABLE].copy()
        n_cps = len(shared_cps)
    else:
        for grp, d in dirs.items():
            assert d.is_dir(), f"missing pooled seed run: {d}"
            assert (d / 'interaction_scores.csv').is_file(), \
                f"missing interaction_scores.csv in {d}"
        gs_s = load_cpdb_matrix(dirs["good"] / 'interaction_scores.csv')
        ps_s = load_cpdb_matrix(dirs["poor"] / 'interaction_scores.csv')
        gs_s, ps_s, cps_s = align_matrices(gs_s, ps_s)
        n_cps = len(cps_s)
        res = wilcoxon_per_interaction(gs_s, ps_s, cps_s, label='score')
        assert len(res) > 0, f"seed {seed}: empty Wilcoxon family"
        res['analysis_scope'] = ANALYSIS_SCOPE
        res['bh_corrected'] = True

    res = res.copy()
    res['downsample_seed'] = seed
    path = out_dir / SEED_DE_TABLE
    res.to_csv(path, index=False)
    seed_written[seed] = path

    # 06_ resolves seed tables by a glob; more than one match is an error there.
    matches = sorted(out_dir.glob('*per_interaction*.csv'))
    assert len(matches) == 1, \
        f"seed {seed}: {len(matches)} files match *per_interaction*.csv in {out_dir}"

    seed_report.append({
        'downsample_seed': seed,
        'good_dir': str(dirs["good"]), 'poor_dir': str(dirs["poor"]),
        'n_cellpairs_shared': int(n_cps),
        'family_size': int(len(res)),
        'n_fdr_lt_threshold': int((res['fdr'] < FDR_THRESHOLD).sum()),
        'is_main_run': seed == 0,
        'analysis_scope': ANALYSIS_SCOPE,
        'path': str(path),
    })

seed_report_df = pd.DataFrame(seed_report)
seed_report_df.to_csv(EXPLORATORY_DIR / 'pooled_seed_de_report.csv', index=False)

assert len(seed_written) >= 2, "downsampling stability needs at least two seed tables"
# Seed 0 must be byte-identical in content to the primary pooled table.
_s0 = pd.read_csv(seed_written[0]).drop(columns=['downsample_seed'])
_p0 = DE_TABLES[PRIMARY_POOLED_TABLE]
assert len(_s0) == len(_p0) and np.allclose(_s0['pvalue'], _p0['pvalue']), \
    "seed_0 table disagrees with the primary pooled table"

_log(event="pooled_seed_differential_enrichment",
     analysis_scope=ANALYSIS_SCOPE,
     interpretation="sensitivity to cell downsampling; NOT patient-level validity",
     seeds=sorted(seed_written),
     report=seed_report,
     output_sha256={str(p): sha256_file(p) for p in seed_written.values()})

seed_report_df

---
## Ordering hazard — differential enrichment must precede consistency

The April 2026 results were wrong for exactly this reason. The consistency matrices
were computed at 14:58 against a differential-enrichment output that was then
overwritten at 15:56. The matrices therefore described a hit list that no longer
existed: **146 interactions vs the 138 that were actually in the final file, and
`higher_in_poor` 106 vs the final 80.** No error was raised; the numbers simply did not
correspond to each other.

The guard below is mechanical, not procedural:

* the mtime of every differential-enrichment output is captured;
* the consistency step records its own start time;
* an assertion requires every DE output to be **strictly older** than the consistency
  start, and both timestamps are written to the run log.

Re-running only the consistency section without re-running the DE section is safe.
Re-running only the DE section after consistency will trip the assertion on the next
consistency run.

In [ ]:
# =============================================================================
# ORDERING GUARD — record DE output mtimes, then open the consistency window.
# =============================================================================
de_mtimes = {name: os.path.getmtime(path) for name, path in de_written.items()}
de_mtimes_iso = {n: datetime.fromtimestamp(m, timezone.utc).isoformat()
                 for n, m in de_mtimes.items()}

CONSISTENCY_START = time.time()
CONSISTENCY_START_ISO = datetime.fromtimestamp(CONSISTENCY_START, timezone.utc).isoformat()

if ENFORCE_DE_BEFORE_CONSISTENCY:
    newest_de = max(de_mtimes.values())
    assert newest_de < CONSISTENCY_START, (
        "ORDERING VIOLATION: a differential-enrichment output is not older than the "
        "consistency run start. Re-run the differential-enrichment cells first.\n"
        f"  newest DE mtime : {datetime.fromtimestamp(newest_de, timezone.utc).isoformat()}\n"
        f"  consistency start: {CONSISTENCY_START_ISO}"
    )

pooled_hits_path = de_written[PRIMARY_POOLED_TABLE]
assert pooled_hits_path.is_file()

_log(event="ordering_guard",
     de_output_mtimes_utc=de_mtimes_iso,
     consistency_start_utc=CONSISTENCY_START_ISO,
     de_strictly_older_than_consistency=True,
     primary_pooled_table_path=str(pooled_hits_path),
     primary_pooled_table_sha256=sha256_file(pooled_hits_path))

de_mtimes_iso

---
## Part 2 — Per-sample consistency

Port of `legacy_sources/cpdb_persample_consistency.py`.

### The one fix that matters

The legacy script hardcodes its sample groups at **lines 58–67**:

```python
GOOD_OUTCOME_SAMPLES = [ ... 17 entries ... ]
POOR_OUTCOME_SAMPLES = [ ... 7 entries ... ]
```

Those lists were derived from the **old `model_outcome` column** and give a 17/7 split.
The current analysis is based on `clinical_outcome`, which gives **13 Good / 11 Poor**.
The lists are consumed at two sites:

* **line 325** — `all_samples = GOOD_OUTCOME_SAMPLES + POOR_OUTCOME_SAMPLES`, which
  determines which patient directories are read at all;
* **lines 203–204** — the `good_cols` / `poor_cols` selection inside
  `compute_consistency_summary`, which determines which columns are averaged into each
  group.

Both lists are deleted here and replaced by a runtime derivation from
`obs['clinical_outcome']`. **Both consumption sites read the derived lists.** If this
substitution were missed the analysis would run to completion and produce silently
wrong results: no error, no warning, just a Good group containing four Poor patients.
`Available: 24 / 24` and the 13/11 split are asserted below.

### Known defect in the legacy matrices — preserved deliberately, not fixed

`score_sig` imputes a hard `0.0` when an interaction is present in a sample but never
reaches CellPhoneDB significance in any cell-type pair (legacy line 176), while
`score_all` keeps `NaN` when the sample's output file or the interaction row is missing.
Two different meanings of absence — "measured, never significant" and "not measured" —
are therefore mixed inside the same matrix, and any mean taken across a `score_sig` row
treats them identically.

This is **not** corrected here. The legacy matrices are retained for continuity with the
previous analysis only. `03_extract_patient_endpoints.ipynb` and
`04_patient_level_inference.ipynb` handle structural missingness correctly — they never
convert it to zero — and they, not these matrices, carry the patient-level inference.

In [ ]:
# =============================================================================
# Runtime derivation of the Good / Poor patient lists.
# Replaces the hardcoded GOOD_OUTCOME_SAMPLES / POOR_OUTCOME_SAMPLES at legacy
# lines 58-67. NEVER hardcode a patient list.
# =============================================================================
# SAMPLE_COL and OUTCOME_COL come from config.yaml (configuration cell).


def _normalise_outcome_map(obj):
    '''Coerce whatever derive_patient_outcomes returns into {sample_id: outcome}.'''
    if isinstance(obj, dict):
        return dict(obj)
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, pd.DataFrame):
        df = obj.reset_index()

        def _pick(cols, prefs):
            lower = {str(c).lower(): c for c in cols}
            for p in prefs:
                if p.lower() in lower:
                    return lower[p.lower()]
            return None

        # Preference order matters: a reset_index() 'index' column must never
        # win over a properly named sample column.
        id_col = _pick(df.columns, [SAMPLE_COL, "sample_id", "sample", "patient_id",
                                    "patient", "batch", "index"])
        oc_col = _pick(df.columns, ["clinical_outcome", "outcome", "group"])
        assert id_col is not None and oc_col is not None, \
            f"cannot identify id/outcome columns in {list(df.columns)}"
        return dict(zip(df[id_col].astype(str), df[oc_col].astype(str)))
    if isinstance(obj, (tuple, list)) and len(obj) == 2:
        good, poor = obj
        return {**{str(s): "Good" for s in good}, **{str(s): "Poor" for s in poor}}
    raise TypeError(f"unsupported return type from derive_patient_outcomes: {type(obj)}")


# Prefer the mapping already derived and checksummed by 01_build_cpdb_inputs.ipynb;
# fall back to deriving it from the H5AD obs table in backed mode.
_map_candidates = sorted(OUTPUT_ROOT.glob("**/*patient_outcome*.csv")) + \
                  sorted(OUTPUT_ROOT.glob("**/*derived_patient*.csv"))
_map_candidates = [p for p in _map_candidates if p.is_file()]

if _map_candidates:
    OUTCOME_SOURCE = _map_candidates[0]
    outcome_map = _normalise_outcome_map(pd.read_csv(OUTCOME_SOURCE))
else:
    import anndata as ad
    h5ad_path = Path(CFG["input_h5ad"])
    OUTCOME_SOURCE = h5ad_path
    _adata = ad.read_h5ad(h5ad_path, backed="r")
    outcome_map = _normalise_outcome_map(
        _call_flex(derive_patient_outcomes,
                   ((_adata.obs,), {}),
                   ((_adata,), {}),
                   ((_adata.obs,), {"sample_col": SAMPLE_COL, "outcome_col": OUTCOME_COL}),
                   ((_adata,), {"sample_col": SAMPLE_COL, "outcome_col": OUTCOME_COL}),
                   ((h5ad_path,), {}))
    )
    _adata.file.close()

GOOD_OUTCOME_SAMPLES = sorted([s for s, o in outcome_map.items() if o == OUTCOME_LABELS[0]])
POOR_OUTCOME_SAMPLES = sorted([s for s, o in outcome_map.items() if o == OUTCOME_LABELS[1]])

assert set(outcome_map.values()) == set(OUTCOME_LABELS), \
    f"unexpected outcome labels: {sorted(set(outcome_map.values()))}"
assert len(GOOD_OUTCOME_SAMPLES) == EXPECTED_N_GOOD, \
    f"expected {EXPECTED_N_GOOD} Good patients, derived {len(GOOD_OUTCOME_SAMPLES)}"
assert len(POOR_OUTCOME_SAMPLES) == EXPECTED_N_POOR, \
    f"expected {EXPECTED_N_POOR} Poor patients, derived {len(POOR_OUTCOME_SAMPLES)}"
assert len(set(GOOD_OUTCOME_SAMPLES) & set(POOR_OUTCOME_SAMPLES)) == 0, \
    "a patient appears in both outcome groups"
assert len(outcome_map) == EXPECTED_N_PATIENTS

_log(event="patient_outcomes_derived",
     outcome_column=OUTCOME_COL,
     source=str(OUTCOME_SOURCE),
     n_good=len(GOOD_OUTCOME_SAMPLES),
     n_poor=len(POOR_OUTCOME_SAMPLES),
     good=GOOD_OUTCOME_SAMPLES,
     poor=POOR_OUTCOME_SAMPLES,
     hardcoded=False,
     legacy_model_outcome_lists_removed=True)

print(f"Good ({len(GOOD_OUTCOME_SAMPLES)}): {GOOD_OUTCOME_SAMPLES}")
print(f"Poor ({len(POOR_OUTCOME_SAMPLES)}): {POOR_OUTCOME_SAMPLES}")

In [ ]:
# =============================================================================
# PORTED from legacy_sources/cpdb_persample_consistency.py
# Matrix construction (score_sig / score_all / binary / nsig) is VERBATIM.
# The only change is that the group lists are passed in rather than read from
# module-level constants (legacy lines 203-204).
# =============================================================================

def load_pooled_hits(filepath, fdr_threshold):
    '''Load significant interactions from the pooled exploratory analysis.'''
    df = pd.read_csv(filepath)
    sig = df[df['fdr'] < fdr_threshold].copy()
    return df, sig


def load_sample_matrix(sample_dir, sample_id, filename):
    '''Load a CellPhoneDB matrix for a single sample. Returns None if missing.'''
    filepath = Path(sample_dir) / sample_id / filename
    if not filepath.exists():
        return None
    return pd.read_csv(filepath)


def get_cellpair_cols(df):
    return [c for c in df.columns if '|' in c]


def extract_sample_data(sample_dir, sample_ids, hit_interaction_ids, cpdb_pval_cutoff):
    '''VERBATIM from legacy lines 100-188.

    NOTE (documented, deliberately not fixed): score_sig imputes 0.0 for
    "present but never significant" while score_all keeps NaN for missing files.
    '''
    score_sig_data = {}   # mean score across CellPhoneDB-significant pairs only
    score_all_data = {}   # mean score across ALL cell-type pairs
    binary_data = {}
    nsig_data = {}

    for sid in sample_ids:
        scores_df = load_sample_matrix(sample_dir, sid, 'interaction_scores.csv')
        pvals_df  = load_sample_matrix(sample_dir, sid, 'pvalues.csv')

        if scores_df is None or pvals_df is None:
            score_sig_data[sid] = {iid: np.nan for iid in hit_interaction_ids}
            score_all_data[sid] = {iid: np.nan for iid in hit_interaction_ids}
            binary_data[sid]    = {iid: np.nan for iid in hit_interaction_ids}
            nsig_data[sid]      = {iid: np.nan for iid in hit_interaction_ids}
            continue

        scores_df = scores_df.set_index('id_cp_interaction')
        pvals_df  = pvals_df.set_index('id_cp_interaction')

        cp_cols_scores = get_cellpair_cols(scores_df)
        cp_cols_pvals  = get_cellpair_cols(pvals_df)

        sample_score_sig = {}
        sample_score_all = {}
        sample_binary = {}
        sample_nsig = {}

        shared_cp = sorted(set(cp_cols_scores) & set(cp_cols_pvals))

        for iid in hit_interaction_ids:
            if iid not in scores_df.index or iid not in pvals_df.index:
                sample_score_sig[iid] = np.nan
                sample_score_all[iid] = np.nan
                sample_binary[iid]    = np.nan
                sample_nsig[iid]      = np.nan
                continue

            pval_row = pvals_df.loc[iid, shared_cp]
            if isinstance(pval_row, pd.DataFrame):
                pval_row = pval_row.iloc[0]
            pval_vals = pval_row.values.astype(float)

            score_row = scores_df.loc[iid, shared_cp]
            if isinstance(score_row, pd.DataFrame):
                score_row = score_row.iloc[0]
            score_vals = score_row.values.astype(float)

            sig_mask = pval_vals < cpdb_pval_cutoff
            n_sig = int(np.sum(sig_mask))

            sample_binary[iid] = 1 if n_sig > 0 else 0
            sample_nsig[iid]   = n_sig

            sample_score_all[iid] = np.mean(score_vals)

            if n_sig > 0:
                sample_score_sig[iid] = np.mean(score_vals[sig_mask])
            else:
                sample_score_sig[iid] = 0.0

        score_sig_data[sid] = sample_score_sig
        score_all_data[sid] = sample_score_all
        binary_data[sid]    = sample_binary
        nsig_data[sid]      = sample_nsig

    score_sig_matrix = pd.DataFrame(score_sig_data, index=hit_interaction_ids)
    score_all_matrix = pd.DataFrame(score_all_data, index=hit_interaction_ids)
    binary_matrix    = pd.DataFrame(binary_data,    index=hit_interaction_ids)
    nsig_matrix      = pd.DataFrame(nsig_data,      index=hit_interaction_ids)

    return score_sig_matrix, score_all_matrix, binary_matrix, nsig_matrix


def compute_consistency_summary(pooled_hits, score_sig_matrix, score_all_matrix,
                                binary_matrix, nsig_matrix,
                                good_samples, poor_samples):
    '''VERBATIM from legacy lines 195-300, except that good_samples / poor_samples
    are function arguments (legacy lines 203-204 read module-level constants
    derived from the retired model_outcome column).'''
    good_cols = [s for s in good_samples if s in score_sig_matrix.columns]
    poor_cols = [s for s in poor_samples if s in score_sig_matrix.columns]

    records = []

    for _, row in pooled_hits.iterrows():
        iid = row['id_cp_interaction']

        if iid not in score_sig_matrix.index:
            continue

        good_scores_sig = score_sig_matrix.loc[iid, good_cols].dropna().values.astype(float)
        poor_scores_sig = score_sig_matrix.loc[iid, poor_cols].dropna().values.astype(float)
        good_scores_all = score_all_matrix.loc[iid, good_cols].dropna().values.astype(float)
        poor_scores_all = score_all_matrix.loc[iid, poor_cols].dropna().values.astype(float)

        good_binary = binary_matrix.loc[iid, good_cols].dropna().values.astype(float)
        poor_binary = binary_matrix.loc[iid, poor_cols].dropna().values.astype(float)

        good_nsig = nsig_matrix.loc[iid, good_cols].dropna().values.astype(float)
        poor_nsig = nsig_matrix.loc[iid, poor_cols].dropna().values.astype(float)

        frac_good = np.mean(good_binary) if len(good_binary) > 0 else np.nan
        frac_poor = np.mean(poor_binary) if len(poor_binary) > 0 else np.nan

        mean_sig_good = np.mean(good_scores_sig) if len(good_scores_sig) > 0 else np.nan
        mean_sig_poor = np.mean(poor_scores_sig) if len(poor_scores_sig) > 0 else np.nan
        mean_all_good = np.mean(good_scores_all) if len(good_scores_all) > 0 else np.nan
        mean_all_poor = np.mean(poor_scores_all) if len(poor_scores_all) > 0 else np.nan

        mean_nsig_good = np.mean(good_nsig) if len(good_nsig) > 0 else np.nan
        mean_nsig_poor = np.mean(poor_nsig) if len(poor_nsig) > 0 else np.nan

        pooled_direction = row.get('direction', '')

        if pooled_direction == 'higher_in_poor':
            score_consistent   = mean_all_poor > mean_all_good
            presence_ok        = frac_poor >= frac_good
            consistent = score_consistent and presence_ok
        elif pooled_direction == 'higher_in_good':
            score_consistent   = mean_all_good > mean_all_poor
            presence_ok        = frac_good >= frac_poor
            consistent = score_consistent and presence_ok
        else:
            consistent = False

        if pooled_direction == 'higher_in_poor':
            breadth_consistent   = mean_nsig_poor > mean_nsig_good
            magnitude_consistent = mean_sig_poor > mean_sig_good
        elif pooled_direction == 'higher_in_good':
            breadth_consistent   = mean_nsig_good > mean_nsig_poor
            magnitude_consistent = mean_sig_good > mean_sig_poor
        else:
            breadth_consistent   = False
            magnitude_consistent = False

        if breadth_consistent and magnitude_consistent:
            signal_type = 'breadth_and_magnitude'
        elif breadth_consistent:
            signal_type = 'breadth_only'
        elif magnitude_consistent:
            signal_type = 'magnitude_only'
        else:
            signal_type = 'neither'

        records.append({
            'id_cp_interaction':          iid,
            'interacting_pair':           row.get('interacting_pair', ''),
            'classification':             row.get('classification', ''),
            'pooled_direction':           pooled_direction,
            'pooled_fdr':                 row.get('fdr', np.nan),
            'pooled_log2fc':              row.get('log2fc_good_vs_poor', np.nan),
            'n_good_samples':             len(good_binary),
            'n_poor_samples':             len(poor_binary),
            'frac_present_good':          frac_good,
            'frac_present_poor':          frac_poor,
            'mean_score_all_good':        mean_all_good,
            'mean_score_all_poor':        mean_all_poor,
            'mean_score_sig_good':        mean_sig_good,
            'mean_score_sig_poor':        mean_sig_poor,
            'mean_nsig_pairs_good':       mean_nsig_good,
            'mean_nsig_pairs_poor':       mean_nsig_poor,
            'consistent_with_pooled':     consistent,
            'signal_type':                signal_type,
        })

    return pd.DataFrame(records)

In [ ]:
# =============================================================================
# Run the consistency check against the FINALISED differential-enrichment output
# =============================================================================
# Consumption site 1 (legacy line 325): all_samples = GOOD + POOR, derived, not hardcoded.
all_samples = GOOD_OUTCOME_SAMPLES + POOR_OUTCOME_SAMPLES
assert len(all_samples) == EXPECTED_N_PATIENTS

pooled_all, pooled_hits = load_pooled_hits(pooled_hits_path, FDR_THRESHOLD)
hit_ids = pooled_hits['id_cp_interaction'].tolist()
assert len(hit_ids) == len(set(hit_ids)), "duplicate interaction ids among pooled hits"

available = [s for s in all_samples if (PER_PATIENT_DIR / s).is_dir()]
missing   = [s for s in all_samples if s not in available]
assert len(available) == EXPECTED_N_PATIENTS, \
    f"Available: {len(available)} / {len(all_samples)} - missing per-patient runs: {missing}"
assert not missing

score_sig_matrix, score_all_matrix, binary_matrix, nsig_matrix = extract_sample_data(
    PER_PATIENT_DIR, available, hit_ids, CPDB_PVAL_CUTOFF,
)

assert list(score_sig_matrix.columns) == available
assert score_all_matrix.shape == score_sig_matrix.shape == binary_matrix.shape == nsig_matrix.shape

# Consumption site 2 (legacy lines 203-204): group column selection, derived.
consistency_summary = compute_consistency_summary(
    pooled_hits, score_sig_matrix, score_all_matrix, binary_matrix, nsig_matrix,
    good_samples=GOOD_OUTCOME_SAMPLES, poor_samples=POOR_OUTCOME_SAMPLES,
)
assert len(consistency_summary) > 0, "consistency summary is empty"
assert (consistency_summary['n_good_samples'] <= EXPECTED_N_GOOD).all()
assert (consistency_summary['n_poor_samples'] <= EXPECTED_N_POOR).all()

id_to_name = dict(zip(pooled_hits['id_cp_interaction'], pooled_hits['interacting_pair']))
consistency_written = {}

for mat, fname in [
    (score_sig_matrix, 'sample_score_sig_matrix.csv'),
    (score_all_matrix, 'sample_score_all_matrix.csv'),
    (binary_matrix,    'sample_binary_matrix.csv'),
    (nsig_matrix,      'sample_nsig_matrix.csv'),
]:
    out = mat.copy()
    out.insert(0, 'interacting_pair', out.index.map(id_to_name))
    path = CONSISTENCY_OUTPUT_DIR / fname
    out.to_csv(path, index_label='id_cp_interaction')
    consistency_written[fname] = path

consistency_summary_out = consistency_summary.copy()
consistency_summary_out['analysis_scope'] = ANALYSIS_SCOPE
consistency_summary_out['pooled_source_table'] = PRIMARY_POOLED_TABLE
_p = CONSISTENCY_OUTPUT_DIR / 'consistency_summary.csv'
consistency_summary_out.to_csv(_p, index=False)
consistency_written['consistency_summary.csv'] = _p

# Group-membership provenance travels with the results.
pd.DataFrame({'sample_id': all_samples,
              'clinical_outcome': ['Good'] * len(GOOD_OUTCOME_SAMPLES) +
                                  ['Poor'] * len(POOR_OUTCOME_SAMPLES),
              'per_patient_run_available': [s in available for s in all_samples]}
             ).to_csv(CONSISTENCY_OUTPUT_DIR / 'patient_group_assignment.csv', index=False)

n_consistent = int(consistency_summary['consistent_with_pooled'].sum())
_log(event="consistency_complete",
     analysis_scope=ANALYSIS_SCOPE,
     consistency_start_utc=CONSISTENCY_START_ISO,
     de_output_mtimes_utc=de_mtimes_iso,
     n_available_patients=len(available),
     n_pooled_tested=int(len(pooled_all)),
     n_pooled_hits=int(len(pooled_hits)),
     n_higher_in_good=int((pooled_hits['direction'] == 'higher_in_good').sum()),
     n_higher_in_poor=int((pooled_hits['direction'] == 'higher_in_poor').sum()),
     n_checked=int(len(consistency_summary)),
     n_consistent=n_consistent,
     signal_type_counts=consistency_summary['signal_type'].value_counts().to_dict(),
     outputs={k: str(v) for k, v in consistency_written.items()})

print(f"Available: {len(available)} / {len(all_samples)}")
print(f"Pooled tested: {len(pooled_all)}   hits at FDR < {FDR_THRESHOLD}: {len(pooled_hits)}")
print(f"  higher_in_good: {(pooled_hits['direction'] == 'higher_in_good').sum()}")
print(f"  higher_in_poor: {(pooled_hits['direction'] == 'higher_in_poor').sum()}")
print(f"Consistent with pooled direction: {n_consistent} / {len(consistency_summary)}")

---
## Part 3 — Concordance annotation against the patient-level results

Both analyses are already finalised before this join happens. Nothing computed here
changes either one.

* The pooled hits **did not** define the patient-level tested family. The primary family
  in `04_` is every endpoint that passes the prespecified 6/6 evaluability and prevalence
  rules, and BH was applied there across that complete family.
* The keys do not match one-to-one. The pooled per-interaction table has one row per
  `id_cp_interaction`; the patient-level table has one row per
  `id_cp_interaction × sender × receiver`. The patient-level effects are therefore
  aggregated to the interaction level (median rank-biserial correlation across that
  interaction's endpoints) purely so the two can be placed side by side.
* What is reported: **direction concordance** and **Spearman rank correlation** where
  keys align.
* What is **not** claimed: independent replication, validation, or confirmation. The two
  analyses use overlapping cells from the same 24 patients; agreement between them is
  expected under the null and carries no inferential weight.

In [ ]:
# =============================================================================
# Concordance annotation (both analyses already final; annotation only)
# =============================================================================
assert PRIMARY_RESULTS_CSV.is_file(), (
    f"patient-level primary results not found at {PRIMARY_RESULTS_CSV}. "
    "Run 04_patient_level_inference.ipynb before the concordance section."
)
primary = pd.read_csv(PRIMARY_RESULTS_CSV)
if 'tested' in primary.columns:
    primary_tested = primary[primary['tested'].astype(bool)].copy()
else:
    primary_tested = primary.copy()
assert len(primary_tested) > 0

CONCORDANCE_NOTE = ("annotation only; overlapping cells from the same 24 patients; "
                    "NOT independent replication and NOT validation")

# --- interaction-level ------------------------------------------------------
pat_int = (primary_tested
           .groupby('id_cp_interaction')
           .agg(patient_median_rank_biserial=('rank_biserial', 'median'),
                patient_min_q=('q_value', 'min'),
                patient_n_endpoints=('rank_biserial', 'size'))
           .reset_index())

pooled_int = DE_TABLES[PRIMARY_POOLED_TABLE][
    ['id_cp_interaction', 'interacting_pair', 'log2fc_good_vs_poor',
     'pvalue', 'fdr', 'direction']
].rename(columns={'pvalue': 'pooled_pvalue', 'fdr': 'pooled_fdr',
                  'direction': 'pooled_direction',
                  'log2fc_good_vs_poor': 'pooled_log2fc'})

conc_int = pooled_int.merge(pat_int, on='id_cp_interaction', how='inner')
conc_int['patient_direction'] = np.where(conc_int['patient_median_rank_biserial'] > 0,
                                         'higher_in_good',
                                         'higher_in_poor')
conc_int['direction_concordant'] = conc_int['pooled_direction'] == conc_int['patient_direction']
conc_int['analysis_scope'] = 'pooled_vs_patient_concordance_annotation'
conc_int['claim'] = CONCORDANCE_NOTE
conc_int.to_csv(EXPLORATORY_DIR / 'pooled_patient_concordance_interaction.csv', index=False)

if len(conc_int) >= 3:
    rho_int, p_int = spearmanr(conc_int['pooled_log2fc'],
                               conc_int['patient_median_rank_biserial'])
else:
    rho_int, p_int = (np.nan, np.nan)
frac_int = float(conc_int['direction_concordant'].mean()) if len(conc_int) else np.nan

# --- cell-pair level --------------------------------------------------------
pat_cp = (primary_tested
          .assign(cell_type_pair=lambda d: d['sender'].astype(str) + '|' + d['receiver'].astype(str))
          .groupby('cell_type_pair')
          .agg(patient_median_rank_biserial=('rank_biserial', 'median'),
               patient_min_q=('q_value', 'min'),
               patient_n_endpoints=('rank_biserial', 'size'))
          .reset_index())

pooled_cp = DE_TABLES['wilcoxon_per_cellpair_scores.csv'][
    ['cell_type_pair', 'log2fc_good_vs_poor', 'pvalue', 'fdr', 'direction']
].rename(columns={'pvalue': 'pooled_pvalue', 'fdr': 'pooled_fdr',
                  'direction': 'pooled_direction',
                  'log2fc_good_vs_poor': 'pooled_log2fc'})

conc_cp = pooled_cp.merge(pat_cp, on='cell_type_pair', how='inner')
conc_cp['patient_direction'] = np.where(conc_cp['patient_median_rank_biserial'] > 0,
                                        'higher_in_good', 'higher_in_poor')
conc_cp['direction_concordant'] = conc_cp['pooled_direction'] == conc_cp['patient_direction']
conc_cp['analysis_scope'] = 'pooled_vs_patient_concordance_annotation'
conc_cp['claim'] = CONCORDANCE_NOTE
conc_cp.to_csv(EXPLORATORY_DIR / 'pooled_patient_concordance_cellpair.csv', index=False)

if len(conc_cp) >= 3:
    rho_cp, p_cp = spearmanr(conc_cp['pooled_log2fc'],
                             conc_cp['patient_median_rank_biserial'])
else:
    rho_cp, p_cp = (np.nan, np.nan)
frac_cp = float(conc_cp['direction_concordant'].mean()) if len(conc_cp) else np.nan

concordance = {
    "interaction_level": {"n_keys_aligned": int(len(conc_int)),
                          "direction_concordance": frac_int,
                          "spearman_rho": None if pd.isna(rho_int) else float(rho_int),
                          "spearman_p": None if pd.isna(p_int) else float(p_int)},
    "cellpair_level": {"n_keys_aligned": int(len(conc_cp)),
                       "direction_concordance": frac_cp,
                       "spearman_rho": None if pd.isna(rho_cp) else float(rho_cp),
                       "spearman_p": None if pd.isna(p_cp) else float(p_cp)},
    "claim": CONCORDANCE_NOTE,
    "pooled_hits_used_to_define_primary_family": False,
}
_log(event="concordance_annotation", **concordance)
concordance

In [ ]:
# =============================================================================
# SUCCESS RECORD
# =============================================================================
all_outputs = sorted(set(list(de_written.values()) + list(consistency_written.values())
                        + list(seed_written.values()) + [
    EXPLORATORY_DIR / 'pooled_seed_de_report.csv',
    DE_OUTPUT_DIR / 'multiplicity_family_report.csv',
    CONSISTENCY_OUTPUT_DIR / 'patient_group_assignment.csv',
    EXPLORATORY_DIR / 'pooled_patient_concordance_interaction.csv',
    EXPLORATORY_DIR / 'pooled_patient_concordance_cellpair.csv',
]))
for p in all_outputs:
    assert p.is_file() and p.stat().st_size > 0, f"missing or empty output: {p}"
    assert str(p).startswith(str(OUTPUT_ROOT)), f"output escaped the new output root: {p}"

_call_flex(write_manifest,
           ((all_outputs, MANIFEST_PATH), {}),
           ((MANIFEST_PATH, all_outputs), {}),
           ((EXPLORATORY_DIR, MANIFEST_PATH), {}),
           ((MANIFEST_PATH,), {"paths": all_outputs}))

SUCCESS = {
    "notebook": NOTEBOOK,
    "status": "success",
    "analysis_scope": ANALYSIS_SCOPE,
    "confirmatory": False,
    "run_start_utc": RUN_START_ISO,
    "run_end_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.time() - RUN_START, 2),
    "config_sha256": CFG_SHA,
    "outcome_column": OUTCOME_COL,
    "outcome_source": str(OUTCOME_SOURCE),
    "n_good_patients": len(GOOD_OUTCOME_SAMPLES),
    "n_poor_patients": len(POOR_OUTCOME_SAMPLES),
    "good_patients": GOOD_OUTCOME_SAMPLES,
    "poor_patients": POOR_OUTCOME_SAMPLES,
    "patient_lists_hardcoded": False,
    "per_patient_runs_available": f"{len(available)}/{EXPECTED_N_PATIENTS}",
    "primary_pooled_table": PRIMARY_POOLED_TABLE,
    "prespecified_before_results": True,
    "family_sizes": de_family_sizes,
    "pooled_seed_tables": {str(k): str(v) for k, v in seed_written.items()},
    "pooled_seed_report": seed_report,
    "bh_corrected_tables": BH_CORRECTED_TABLES,
    "uncorrected_tables": UNCORRECTED_TABLES,
    "de_output_mtimes_utc": de_mtimes_iso,
    "consistency_start_utc": CONSISTENCY_START_ISO,
    "ordering_guard_passed": True,
    "score_sig_zero_imputation_defect": "present, preserved deliberately, documented",
    "concordance": concordance,
    "outputs": [str(p) for p in all_outputs],
    "output_sha256": {str(p): sha256_file(p) for p in all_outputs},
    "manifest": str(MANIFEST_PATH),
}
SUCCESS_JSON.write_text(json.dumps(SUCCESS, indent=2))
_log(event="notebook_success", **{k: v for k, v in SUCCESS.items() if k != "output_sha256"})
SUCCESS